# Notebook 04 — Customer Segmentation
**Goal:** Identify distinct customer segments using K-Means clustering.
Profile each segment by conversion rate, demographics, and contact behaviour.
Map segments to campaign targeting recommendations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')
from segmentation import (
    prepare_features, elbow_plot, fit_kmeans,
    pca_scatter, segment_profile, segment_conv_bar
)
from eda_utils import save

df = pd.read_csv('../data/processed/cleaned_data.csv')
print(f'Records: {len(df):,}')

## 1. Feature Preparation

In [ ]:
X_scaled, feature_names, scaler = prepare_features(df)
print(f'Feature matrix shape: {X_scaled.shape}')
print(f'Features used: {feature_names[:8]}... ({len(feature_names)} total)')

## 2. Optimal K — Elbow + Silhouette

In [ ]:
fig, inertias, silhouettes = elbow_plot(X_scaled, k_range=range(2, 9))
save(fig, '12_elbow_silhouette.png')
plt.show()

best_k = list(range(2,9))[silhouettes.index(max(silhouettes))]
print(f'Best K by silhouette score: {best_k} (score={max(silhouettes):.4f})')

## 3. Fit K-Means (k=4)

In [ ]:
km, labels = fit_kmeans(X_scaled, k=4)
df['segment'] = labels + 1
print(f'\nSegment size distribution:')
print(df['segment'].value_counts().sort_index())

## 4. PCA Scatter — Visual Segment Separation

In [ ]:
fig = pca_scatter(X_scaled, labels)
save(fig, '13_pca_segments.png')
plt.show()

## 5. Segment Profiles

In [ ]:
profile = segment_profile(df, labels)
print('\nSegment Profile:')
print(profile.to_string(index=False))

## 6. Conversion Rate by Segment

In [ ]:
fig = segment_conv_bar(profile)
save(fig, '14_segment_conversion.png')
plt.show()

## 7. Segment Heatmap — Feature Means

In [ ]:
num_cols = ['age','campaign','previous','euribor3m','cons_conf_idx','subscribed']
heat = df.groupby('segment')[num_cols].mean()

fig, ax = plt.subplots(figsize=(10,4))
norm = (heat - heat.min()) / (heat.max() - heat.min())
sns.heatmap(norm.T, annot=heat.T.round(2), fmt='.2f',
            cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Segment Feature Heatmap (Normalized)', fontsize=13, fontweight='bold')
ax.set_xticklabels([f'Seg {i}' for i in heat.index])
plt.tight_layout()
save(fig, '15_segment_heatmap.png')
plt.show()

## 8. Save Segmented Data & Business Labels

In [ ]:
# Assign business-friendly names based on profile
# (Update mapping after reviewing actual profile output)
seg_names = {
    profile.sort_values('conv_rate', ascending=False).iloc[0]['segment']: 'High-Value Responders',
    profile.sort_values('conv_rate', ascending=False).iloc[1]['segment']: 'Warm Prospects',
    profile.sort_values('conv_rate', ascending=False).iloc[2]['segment']: 'Price Sensitive',
    profile.sort_values('conv_rate', ascending=False).iloc[3]['segment']: 'Hard to Convert',
}
df['segment_name'] = df['segment'].map(seg_names)
df.to_csv('../data/processed/segmented_data.csv', index=False)
print('Saved → data/processed/segmented_data.csv')
print('\nSegment name distribution:')
print(df['segment_name'].value_counts())